# <a id='toc1_'></a>[Using gene symbol categories to resolve ambiguous gene symbols](#toc0_)

Gene symbol categories are determined by the relationship the gene symbol has with its associated gene concept. These categories are ranked to show priority and used to assign ambiguous gene symbols to unique gene concepts.

This notebook does two things:
1. Collects a set of ambiguous gene symbols that have previously been annotated to unique gene concepts
2. Validates a ranked list of gene symbol categories against the set of ambiguous gene symbols

**Table of contents**<a id='toc0_'></a>    
- [Using gene symbol categories to resolve ambiguous gene symbols](#toc1_)    
  - [Collect a set of ambiguous gene symbols that have been annotated to unique gene concepts](#toc1_1_)    
    - [CIViC](#toc1_1_1_)    
      - [Build lookup by CIViC molecular profile ID](#toc1_1_1_1_)    
      - [Make an ambiguous symbol dataframe using alias-alias and alias-primary collisions](#toc1_1_1_2_)    
      - [Remove genes from civic_df that are not involved in collisions (don't have associated ambiguous gene symbols)](#toc1_1_1_3_)    
      - [Query articles for ambiguous symbol using Pubtator3](#toc1_1_1_4_)    
      - [Only 23 documents and 5 gene-symbol pairs, need more](#toc1_1_1_5_)    
    - [DGIdb](#toc1_1_2_)    
    - [James provided raw data of all gene claims](#toc1_1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import sys
from pathlib import Path

analysis_dir = Path.cwd().parent

if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import importlib
import re

import civicpy.civic as civic
import gene_ids_in_lit.functions as giilfn
import collision_analysis_shared_variables as casv
import pandas as pd
import functions as fn

importlib.reload(giilfn)
importlib.reload(fn)

<module 'functions' from '/Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/ranked_category_resolver/functions.py'>

## <a id='toc1_1_'></a>[Collect a set of ambiguous gene symbols that have been annotated to unique gene concepts](#toc0_)

### <a id='toc1_1_2_'></a>[Using DGIdb gene claims](#toc0_)

James (DGIdb developer) provided raw data of all gene claims, the website download of genes.tsv does not include the "aliases" column that includes supporting information provided by the source

In [2]:
dgidb_genes_df = pd.read_csv("../../input/gene_claims20260814.csv")

In [3]:
# Number of gene claims

len(dgidb_genes_df)

79767

#### Filter the DGIdb gene claims so that only those using alias gene symbols remain

In [4]:
alias_dgidb_genes_df = dgidb_genes_df[
    (~dgidb_genes_df["name"].isin(casv.primary_symbol_set)) # never a primary gene symbol
    & (dgidb_genes_df["name"].isin(casv.alias_symbol_set))
].copy()

In [5]:
len(alias_dgidb_genes_df)

752

#### Filter the dgidb gene claims so that only those using ambiguous gene symbols remain

In [6]:
# Collision dfs were generated in notebooks 1 and 2

aa_collision_df = pd.read_csv("../../output/merged_aa_collision_gene_df.csv")

aa_collision_df = aa_collision_df.rename(columns={
    "collision": "ambiguous_symbol",
})
ap_collision_dp = pd.read_csv("../../output/merged_alias_primary_collisions_df.csv")

ap_collision_dp = ap_collision_dp.rename(columns={
    "collision": "ambiguous_symbol",
})

In [7]:
# Assign a collision type to each dataframe to be retained after merging

aa_collision_df["collision type"] = "alias-alias"
ap_collision_dp["collision type"] = "alias-primary"

In [8]:
# Merge alias-alias and alias-primary collision dataframes into one ambiguous symbol dataframe
# Columns inlcude: ambiguous symbol, NCBI ID, primary gene symbol, and collision type

collision_df = pd.concat(
    [aa_collision_df, ap_collision_dp],
    ignore_index=True
)

key_cols = [
    "ambiguous_symbol",
    "NCBI_ID",
    "primary_gene_symbol",
]

collision_df = (
    collision_df
    .groupby(
        key_cols,
        as_index=False,
        dropna=False,
    )["collision type"]
    .agg(lambda x: sorted(set(x)))
)

In [9]:
# Group rows by primary gene symbol and NCBI ID so that ambiguous gene symbols associated to one gene will be in a list in the ambiguous symbol column

collision_df = (
    collision_df
    .groupby(
        ["NCBI_ID", "primary_gene_symbol"],
        as_index=False,
        dropna=False,
    )
    .agg({
        "ambiguous_symbol": lambda x: sorted(set(x)),
        "collision type": lambda x: sorted(
            {item for sublist in x for item in sublist}
        ),
    })
)

In [10]:
# Filter for ambiguous gene claim names

ambiguous_alias_dgidb_genes_df = alias_dgidb_genes_df[
    alias_dgidb_genes_df["name"].isin(
        collision_df["ambiguous_symbol"].explode()
    )
].copy()

In [11]:
# Don't need this column
ambiguous_alias_dgidb_genes_df = ambiguous_alias_dgidb_genes_df.drop(columns=["nomenclature"])

In [12]:
# Removing the source prefix for easier comparison

ambiguous_alias_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_df[ambiguous_alias_dgidb_genes_df["aliases"].notna()]

ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_HGNC_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
    r"(?i)hgnc:([^|]+)",
    expand=False,
)

ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_NCBI_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
    r"(?i)ncbigene:([^|]+)",
    expand=False,
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/3364182606.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_HGNC_ID"] = ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_gene_id"].str.extract(
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/3364182606.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["normalized_NCBI_ID"] = ambiguous_alias_dgidb_genes_

#### This yields 102 gene claims that normalize ambiguous gene symbol to gene concepts

### Validate ranked gene symbol categories

In [13]:
# This file contains gene-to-symbol pairs annotated with relationship categories

capture_df = pd.read_csv("../../output/summary_df.csv")

In [14]:
# Currently the values in the ID columns are just strings that look like sets, need to convert to actual sets

for col in ["HGNC_ID", "NCBI_ID", "ENSG_ID"]:
    capture_df[col] = capture_df[col].apply(fn.to_set)

#### Rank the relationship categories

In [15]:
rank_order = ["Primary Gene Symbol",
                "Previous Symbol",
                "Clone Name Symbol",
                "Gene Identifier Symbol",
                "Placeholder Symbol",
                "Ortholog Symbol",
                "Alternate Abbreviation Symbol",
                "Withdrawn Ortholog Symbol",
                "Prefix Condition Symbol",
                "Gene Group Symbol",
                "Protein Mass Symbol",
                "Related Gene Symbol",
                "Gene Neighbor Symbol",
                "Gene Interaction Symbol"]

rank_map = {category: i for i, category in enumerate(rank_order)}

#### Use the ranked categories to resolve the ambiguous gene symbol to a gene concept

In [16]:
# Run the function on every ambiguous DGIdb row

result_cols = [
    "rank_match",
    "rank_status",
    "winning_category",
    "winning_HGNC_ID"
]

ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
    ambiguous_alias_dgidb_genes_with_identifiers_df.apply(
        fn.check_rank_match,
        axis=1,
        capture_df=capture_df,
        rank_map=rank_map,
    )
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/2566745398.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/2566745398.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df[result_cols] = (
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/2566745398.py:10: SettingWithCopyWarning: 
A value is tryin

In [17]:
# Covert the rank_match type to boolean

ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"] = (
    ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"]
    .astype("boolean")
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_62624/1773379390.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_alias_dgidb_genes_with_identifiers_df["rank_match"] = (


In [18]:
ambiguous_alias_dgidb_genes_with_identifiers_df["rank_status"].value_counts()

rank_status
matched           60
no captured as    21
ID mismatch       21
Name: count, dtype: int64

In [19]:
matched_ambiguous_alias_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_with_identifiers_df[ambiguous_alias_dgidb_genes_with_identifiers_df["rank_status"] == "matched"]
matched_ambiguous_alias_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
153,DAC,DrugBank,hgnc:17,DAC_ACTSP|UNIPROT:P39045|X64790,17,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:17}
599,ARSC,DrugBank,hgnc:716,ARSC_STAAU|ARSC1_ECOLX|GENBANK:150729|GENBANK:...,716,NaN,True,matched,Previous Symbol,{HGNC:716}
3115,CCRL1,dGene,hgnc:1611,CC-CKR-11|CCBP2|CCR-11|CCR10|CCR11|CCX CKR|CCX...,1611,NaN,True,matched,Previous Symbol,{HGNC:1611}
5668,APR,DrugBank,hgnc:6692,GENBANK:142526|GENBANK:5921206|K02496|SUBD_BAC...,6692,NaN,True,matched,Previous Symbol,{HGNC:6692}
5868,DUSP13,dGene,hgnc:19681,BEDP|DUSP13A|DUSP13B|MDSP|NCBIGENE:51207|SKRP4...,19681,NaN,True,matched,Previous Symbol,{HGNC:19681}
7010,CMK,DrugBank,hgnc:7098,GENBANK:42839|KCY_ECOLI|UNIPROT:P0A6I0|X00785,7098,NaN,True,matched,Previous Symbol,{HGNC:7098}
9018,NOS,DrugBank,hgnc:7872,BA000033|D86417|GENBANK:21205025|GENBANK:24432...,7872,NaN,True,matched,Previous Symbol,{HGNC:7872}
9590,MARS,Pharos,hgnc:6898,"METHIONINE--TRNA LIGASE, CYTOPLASMIC|UNIPROT:P...",6898,NaN,True,matched,Previous Symbol,{HGNC:6898}
10096,L5,DrugBank,hgnc:10360,Q64822_9ADEN|Q64823_9ADEN|UNIPROT:Q64822|UNIPR...,10360,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:10360}
15775,PTA,DrugBank,hgnc:21290,GENBANK:580883|PTAS_BACSU|UNIPROT:P39646|X73124,21290,NaN,True,matched,Alternate Abbreviation Symbol,{HGNC:21290}


In [20]:
matched_ambiguous_alias_dgidb_genes_with_identifiers_df["winning_category"].value_counts()

winning_category
Previous Symbol                  44
Alternate Abbreviation Symbol     8
Ortholog Symbol                   4
Gene Group Symbol                 2
Prefix Condition Symbol           1
Withdrawn Ortholog Symbol         1
Name: count, dtype: int64

#### Investigate the gene claims where the ranked categories mapped the ambiguous gene symbol to a different gene concept than the source

In [21]:
mismatched_ambiguous_dgidb_genes_with_identifiers_df = ambiguous_alias_dgidb_genes_with_identifiers_df[ambiguous_alias_dgidb_genes_with_identifiers_df["rank_status"] == "ID mismatch"]
mismatched_ambiguous_dgidb_genes_with_identifiers_df

,name,source_db_name,normalized_gene_id,aliases,normalized_HGNC_ID,normalized_NCBI_ID,rank_match,rank_status,winning_category,winning_HGNC_ID
4020,HCA1,NCBI,ncbigene:266790,"AH|HCA|Hypercalciuria, absorptive, 1|ncbigene:...",NaN,266790,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:4532}
4337,ENV,DrugBank,hgnc:39031,ENV_HV1BN|ENV_HV1Y2|ENV_SIVMK|M21098|UNIPROT:P...,39031,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:53424}
4550,ACT,NCBI,ncbigene:389036,ncbigene:389036,NaN,389036,False,ID mismatch,Gene Group Symbol,{HGNC:17780}
5936,MST,DrugBank,hgnc:29678,AJ313201|Q7K9G0_LEIMA|UNIPROT:Q7K9G0,29678,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:7223}
9728,GL,ChEMBL,hgnc:14942,CHEMBL:CHEMBL2364696|ENVELOPE GLYCOPROTEIN L|U...,14942,NaN,False,ID mismatch,Ortholog Symbol,{HGNC:21652}
13166,MOP,DrugBank,hgnc:14449,GENBANK:853817|MOP_DESGI|UNIPROT:Q46509|X77222,14449,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:8156}
13591,ALR,ChEMBL,hgnc:380,ALANINE RACEMASE|CHEMBL:CHEMBL2031|UNIPROT:P9WQA9,380,NaN,False,ID mismatch,Alternate Abbreviation Symbol,{HGNC:4236}
13980,CAMK,DrugBank,hgnc:1464,AF323755|CAMK_RHOSO|UNIPROT:Q93TU6,1464,NaN,False,ID mismatch,Gene Group Symbol,{HGNC:1463}
16683,PARC,DrugBank,hgnc:10616,GENBANK:147106|GENBANK:1490399|GENBANK:1574370...,10616,NaN,False,ID mismatch,Withdrawn Ortholog Symbol,{HGNC:15982}
16814,TGT,DrugBank,hgnc:12612,GENBANK:498141|L33777|TGT_ZYMMO|UNIPROT:P28720,12612,NaN,False,ID mismatch,Ortholog Symbol,{HGNC:23797}


In [22]:
mismatched_ambiguous_dgidb_genes_with_identifiers_df["winning_category"].value_counts()

winning_category
Alternate Abbreviation Symbol    13
Gene Group Symbol                 3
Ortholog Symbol                   2
Withdrawn Ortholog Symbol         2
Previous Symbol                   1
Name: count, dtype: int64

In [23]:
capture_df[
    capture_df["gene_symbol"].str.upper() == "PAR4"
]

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
32122,32122,{HGNC:3540},{ENSG00000127533},{GENE ID:9002},F2RL3,PAR4,T,Previous Symbol
86518,86518,{HGNC:8614},{ENSG00000177425},{GENE ID:5074},PAWR,PAR4,F,NaN
95886,95886,{HGNC:29998},{},{GENE ID:347745},PWAR4,PAR4,T,Alternate Abbreviation Symbol


In [24]:
mismatched_ambiguous_dgidb_genes_with_identifiers_df.to_csv(
    "output/mismatched_ambiguous_dgidb_genes_with_identifiers_df.csv",
    index=False,
)

In [25]:
manually_annotated_resons_mismatched_ambiguous_dgidb_genes_with_identifiers_df = pd.read_csv("output/manually_annotated_resons_mismatched_ambiguous_dgidb_genes_with_identifiers_df.csv")
manually_annotated_resons_mismatched_ambiguous_dgidb_genes_with_identifiers_df["reason for mismatch"].value_counts()

reason for mismatch
correct concept relationship is undefined    14
missing gene record                           3
same relationship tie                         3
wrong category prioritized                    1
Name: count, dtype: int64